# Plots for the report

Reads the result CSVs and saves 3 PNGs.

In [ ]:
!pip install matplotlib pandas

In [ ]:
import os, pandas as pd
import matplotlib.pyplot as plt
os.makedirs("figures", exist_ok=True)

In [ ]:
# upload fairness_summary.csv, blind_screening_summary.csv, fairness_comparison.csv
from google.colab import files
files.upload()

In [ ]:
fair = pd.read_csv("fairness_summary.csv")
blind = pd.read_csv("blind_screening_summary.csv")

labels = fair["changed_signal"].tolist()
before = fair["average_absolute_difference"].tolist()
after = blind.set_index("changed_signal").loc[labels, "average_blind_absolute_difference"].tolist()

x = range(len(labels))
w = 0.35
plt.figure(figsize=(6,4))
plt.bar([i-w/2 for i in x], before, w, label="before blind")
plt.bar([i+w/2 for i in x], after, w, label="after blind")
plt.xticks(list(x), labels)
plt.ylabel("avg |score diff|")
plt.title("sbert score change per signal")
plt.legend(); plt.tight_layout()
plt.savefig("figures/fig_before_after_blind.png", dpi=200)
plt.show()

In [ ]:
cmp = pd.read_csv("fairness_comparison.csv")
dmap = {"Software Engineer":"Software Engineering","Financial Analyst":"Finance",
        "Marketing Coordinator":"Marketing","Clinical Data Analyst":"Healthcare",
        "Academic Advisor":"Education"}
cmp["domain"] = cmp["job_title"].map(dmap)

by = cmp.groupby(["domain","changed_signal"])["absolute_difference"].mean().unstack()
by.plot(kind="bar", figsize=(7,4))
plt.ylabel("avg |score diff|")
plt.title("score change by domain and signal")
plt.xticks(rotation=20); plt.tight_layout()
plt.savefig("figures/fig_by_domain.png", dpi=200)
plt.show()

In [ ]:
nd = cmp.loc[cmp["changed_signal"]=="name","score_difference"]
plt.figure(figsize=(6,4))
plt.hist(nd, bins=15, edgecolor="black")
plt.axvline(0, color="red", linestyle="--")
plt.xlabel("score difference (changed - original)")
plt.ylabel("pairs")
plt.title("name change: where the scores moved")
plt.tight_layout()
plt.savefig("figures/fig_name_distribution.png", dpi=200)
plt.show()

In [ ]:
from google.colab import files
for f in ["fig_before_after_blind.png","fig_by_domain.png","fig_name_distribution.png"]:
    files.download(f"figures/{f}")